In [3]:
import pandas as pd

#zip file processing
import zipfile

#directory controls
import os

import io

In [18]:
#change to your file directory
ERC_DATA = "..\\datasets\\ERC20-stablecoins.zip"
GFC_DATA="..\\datasets\\gfc.zip"

In [6]:
os.makedirs("../datasets/processed/erc_20", exist_ok=True)  # creates nested directories
os.makedirs("../datasets/processed/gfc_data", exist_ok=True)

In [21]:
def describe(f, name):
    """
    To understand more about the dataframe (columns, num of null values, shape)
    Args:
        f -> String representing : file path
        name -> String : file name
    Output:
        None
    """
    if name.startswith("event_data"):
        df = pd.read_csv(f, encoding='latin-1')
    else:
        df = pd.read_csv(f)
    print(f"\n📋 All columns in {name}")
    print(df.columns.tolist())
    print(f"\n📊 First 5 rows:")
    print(df.head(5))
    print(f"\n Dataframe shape")
    print(df.shape)
    print(f"\n Number of null values")
    print(df.isna().sum())
    return df

# Processing ERC 20 Data

In [ ]:
data_dataframes = {}
with zipfile.ZipFile(ERC_DATA) as z:
    for name in z.namelist():

        if name.endswith(".zip"):
            # if it is another zipped folder
            print(f"Opening nested zip {name}")
            nested_zip = z.read(name)

            with zipfile.ZipFile(io.BytesIO(nested_zip)) as nested_z:
                for fileName in nested_z.namelist():
                    with nested_z.open(fileName) as nested_f:
                        df = pd.read_csv(nested_f)
                        df['date'] = pd.to_datetime(df["timestamp"], unit="s") #convert unix timestamp to date time
                        df["coins"] = fileName.split("_")[0] #get the coin name as column
                        df.to_csv(f"../datasets/processed/erc_20/processed_{fileName.split("/")[1]}", index=False)
        else:
            if ("token" in name):
                continue

            else:
                print(f"Opening file {name}")
                # if it is a file
                with z.open(name) as f:
                    df = describe(f, name)

                    if ("timestamp" in df.columns): #conditions according to column name
                        df["date"] = pd.to_datetime(df["timestamp"], unit="s") #convert unix timestamp to date time
                    elif ("time_stamp" in df.columns):
                        df["date"] = pd.to_datetime(df["time_stamp"], unit="s") #convert unix timestamp to date time
                    df.to_csv(f"../datasets/processed/erc_20/processed_{name}", index=False)

Opening nested zip price_data.zip
Opening file event_data.csv

📋 All columns in event_data.csv
['event', 'timestamp', 'type', 'stablecoin']

📊 First 5 rows:
                                               event   timestamp      type  \
0  BlackRock and Fidelity Back USDC in $400 Milli...  1649721600  positive   
1  Terra UST takes over BUSD to become third larg...  1650412800  positive   
2  LARGE amounts of UST selling on ANCHOR (approx...  1651881600  negative   
3  UST depegs LFG deploys assets to defend peg (7...  1651968000  negative   
4    UST Depegs again to 35 cents LUNA keeps falling  1652054400  negative   

  stablecoin  
0       usdc  
1       ustc  
2       ustc  
3       ustc  
4       ustc  

 Dataframe shape
(38, 4)

 Number of null values
event         0
timestamp     0
type          0
stablecoin    0
dtype: int64


# Processing GFC Data

In [6]:
# for filename in os.listdir(GFC_DATA_FOLDER):
#     if filename.endswith(".csv"):
#         file_path = os.path.join(GFC_DATA_FOLDER, filename)
#         if os.path.isfile(file_path):
#             print(filename)

#             df = describe(file_path, filename) #understand the file
#             print(f"Processing file for {df.iloc[0,1]}")
#             #formating df's format
#             ticker = df.iloc[0, 1] #get ticker value
#             date= df.iloc[2:, 0] # get date column
#             df = df.iloc[3:,:] # get the relevant dataset (from row 2 onwards)
#             df['ticker'] = ticker #assigned ticker to ticker column
#             df['date'] = date #assigned date to date column
#             df.to_csv(f"../datasets/processed/gfc_data/processed_{filename}", index=False)


In [19]:
# format detection
def is_yahoo_style(df):
    """Detect Yahoo multi-row header format."""
    return (
        "Price" in df.columns and
        len(df) >= 2 and
        str(df.iloc[0, 0]) == "Ticker" and
        str(df.iloc[1, 0]) == "Date"
    )


# cleaners
def yahoo_clean(df, filename):
    """Apply Yahoo metadata cleanup."""
    print(f"✨ Yahoo-cleaning → {filename}")

    ticker_val = df.iloc[0, 1]

    df = df.iloc[2:].copy()
    df.rename(columns={"Price": "Date"}, inplace=True)

    df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
    df["ticker"] = ticker_val

    return df


def apply_unified_format(df, filename):
    """Apply shared formatting rules to all files."""
    print(f"📊 Formatting → {filename}")

    # Ensure ticker exists
    if "ticker" not in df.columns:
        if len(df) > 0 and df.shape[1] > 1:
            df["ticker"] = df.iloc[0, 1]
        else:
            df["ticker"] = "UNKNOWN"

    # Standardize date column
    if "Date" in df.columns:
        df["date_standardized"] = pd.to_datetime(df["Date"], errors="coerce")
    else:
        first_col = df.columns[0]
        df["date_standardized"] = pd.to_datetime(df[first_col], errors="coerce")

    df.reset_index(drop=True, inplace=True)

    return df


# file processor
def process_and_save(file_handle, filename, output_folder):

    clean_name = os.path.basename(filename)
    print(f"\n📄 Processing {clean_name}")

    try:
        df = pd.read_csv(file_handle)

        # Step 1 — Yahoo cleanup (if applicable)
        if is_yahoo_style(df):
            df = yahoo_clean(df, clean_name)

        # Step 2 — Unified formatting (always applied)
        df = apply_unified_format(df, clean_name)

        # Save
        save_path = os.path.join(output_folder, f"processed_{clean_name}")
        df.to_csv(save_path, index=False)

        print(f"✅ Saved → {save_path}")

    except Exception as e:
        print(f"❌ Failed → {clean_name}: {e}")


# pipeline runner
def run_unified_pipeline(source_path, output_folder):

    os.makedirs(output_folder, exist_ok=True)

    # ZIP source
    if zipfile.is_zipfile(source_path):
        print(f"📦 ZIP detected: {source_path}")

        with zipfile.ZipFile(source_path, "r") as z:
            for name in z.namelist():
                if name.endswith(".csv"):
                    with z.open(name) as f:
                        process_and_save(f, name, output_folder)

    # Folder source
    elif os.path.isdir(source_path):
        print(f"📂 Folder detected: {source_path}")

        for root, _, files in os.walk(source_path):
            for name in files:
                if name.endswith(".csv"):
                    path = os.path.join(root, name)
                    with open(path, "rb") as f:
                        process_and_save(f, name, output_folder)

    else:
        print("❌ Invalid source path")


# run this line
run_unified_pipeline(GFC_DATA, "../datasets/processed/gfc_data/")

📦 ZIP detected: ..\datasets\gfc.zip

📄 Processing AIG.csv
✨ Yahoo-cleaning → AIG.csv
📊 Formatting → AIG.csv
✅ Saved → ../datasets/processed/gfc_data/processed_AIG.csv

📄 Processing ^VIX.csv
✨ Yahoo-cleaning → ^VIX.csv
📊 Formatting → ^VIX.csv
✅ Saved → ../datasets/processed/gfc_data/processed_^VIX.csv

📄 Processing C.csv
✨ Yahoo-cleaning → C.csv
📊 Formatting → C.csv
✅ Saved → ../datasets/processed/gfc_data/processed_C.csv

📄 Processing JPM.csv
✨ Yahoo-cleaning → JPM.csv
📊 Formatting → JPM.csv
✅ Saved → ../datasets/processed/gfc_data/processed_JPM.csv

📄 Processing ^GSPC.csv
✨ Yahoo-cleaning → ^GSPC.csv
📊 Formatting → ^GSPC.csv
✅ Saved → ../datasets/processed/gfc_data/processed_^GSPC.csv

📄 Processing WGS3MO.csv
📊 Formatting → WGS3MO.csv
✅ Saved → ../datasets/processed/gfc_data/processed_WGS3MO.csv

📄 Processing ^DJI.csv
✨ Yahoo-cleaning → ^DJI.csv
📊 Formatting → ^DJI.csv
✅ Saved → ../datasets/processed/gfc_data/processed_^DJI.csv

📄 Processing TEDRATE.csv
📊 Formatting → TEDRATE.csv
✅ Sa

# Investigating Missing Data
Financial data may be missing due to market closures on public holidays and weekends.

In [22]:
describe("..\\datasets\\processed\\gfc_data\\processed_TEDRATE.csv", "Tedrate")


📋 All columns in Tedrate
['observation_date', 'TEDRATE', 'ticker', 'date_standardized']

📊 First 5 rows:
  observation_date  TEDRATE  ticker date_standardized
0       2005-01-04     0.27    0.27        2005-01-04
1       2005-01-05     0.30    0.27        2005-01-05
2       2005-01-06     0.34    0.27        2005-01-06
3       2005-01-07     0.33    0.27        2005-01-07
4       2005-01-10     0.29    0.27        2005-01-10

 Dataframe shape
(2085, 4)

 Number of null values
observation_date       0
TEDRATE              123
ticker                 0
date_standardized      0
dtype: int64


,observation_date,TEDRATE,ticker,date_standardized
0,2005-01-04,0.27,0.27,2005-01-04
1,2005-01-05,0.30,0.27,2005-01-05
2,2005-01-06,0.34,0.27,2005-01-06
3,2005-01-07,0.33,0.27,2005-01-07
4,2005-01-10,0.29,0.27,2005-01-10
...,...,...,...,...
2080,2012-12-25,NaN,0.27,2012-12-25
2081,2012-12-26,NaN,0.27,2012-12-26
2082,2012-12-27,0.23,0.27,2012-12-27
2083,2012-12-28,0.30,0.27,2012-12-28


In [10]:
# Get rows with null values from TEDRATE
tedrate_df = pd.read_csv("..\\datasets\\processed\\gfc_data\\processed_TEDRATE.csv")
null_rows = tedrate_df[tedrate_df.isna().any(axis=1)]

print(f"Total rows with null values: {len(null_rows)}")
print(f"\nNull rows:\n")
null_rows

Total rows with null values: 123

Null rows:



,observation_date,TEDRATE,ticker,date_standardized
9,2005-01-17,NaN,0.27,2005-01-17
34,2005-02-21,NaN,0.27,2005-02-21
58,2005-03-25,NaN,0.27,2005-03-25
59,2005-03-28,NaN,0.27,2005-03-28
84,2005-05-02,NaN,0.27,2005-05-02
...,...,...,...,...
2040,2012-10-30,NaN,0.27,2012-10-30
2049,2012-11-12,NaN,0.27,2012-11-12
2057,2012-11-22,NaN,0.27,2012-11-22
2080,2012-12-25,NaN,0.27,2012-12-25


In [11]:
# Check if null dates are US public holidays
import holidays

# Create US federal holidays calendar
us_holidays = holidays.US(years=range(2005, 2025))

# Convert date column to datetime
null_rows['date_check'] = pd.to_datetime(null_rows['observation_date'])

# Check which null dates are holidays
null_rows['is_holiday'] = null_rows['date_check'].apply(lambda x: x in us_holidays)
null_rows['holiday_name'] = null_rows['date_check'].apply(lambda x: us_holidays.get(x, 'Not a holiday'))

# Summary
print(f"Total null rows: {len(null_rows)}")
print(f"Null rows on holidays: {null_rows['is_holiday'].sum()}")
print(f"Null rows NOT on holidays: {(~null_rows['is_holiday']).sum()}")
print(f"\nPercentage of nulls on holidays: {null_rows['is_holiday'].sum() / len(null_rows) * 100:.1f}%")

# Show sample of holidays
print(f"\nSample of null dates that are holidays:")
null_rows[null_rows['is_holiday']][['observation_date', 'holiday_name']].head(10)

Total null rows: 123
Null rows on holidays: 77
Null rows NOT on holidays: 46

Percentage of nulls on holidays: 62.6%

Sample of null dates that are holidays:


,observation_date,holiday_name
9,2005-01-17,Martin Luther King Jr. Day
34,2005-02-21,Washington's Birthday
104,2005-05-30,Memorial Day
129,2005-07-04,Independence Day
174,2005-09-05,Labor Day
199,2005-10-10,Columbus Day
223,2005-11-11,Veterans Day
232,2005-11-24,Thanksgiving Day
254,2005-12-26,Christmas Day (observed)
259,2006-01-02,New Year's Day (observed)


In [12]:
# Check if non-holiday nulls are on weekends
non_holiday_nulls = null_rows[~null_rows['is_holiday']].copy()
non_holiday_nulls['day_of_week'] = pd.to_datetime(non_holiday_nulls['observation_date']).dt.day_name()
non_holiday_nulls['is_weekend'] = pd.to_datetime(non_holiday_nulls['observation_date']).dt.dayofweek >= 5

print(f"Null rows NOT on holidays: {len(non_holiday_nulls)}")
print(f"Of these, on weekends: {non_holiday_nulls['is_weekend'].sum()}")
print(f"Of these, on weekdays: {(~non_holiday_nulls['is_weekend']).sum()}")
print(f"\nBreakdown by day of week:")
print(non_holiday_nulls['day_of_week'].value_counts().sort_index())
print(f"\nSample of non-holiday null dates:")
non_holiday_nulls[['observation_date', 'day_of_week', 'is_weekend']].head(10)

Null rows NOT on holidays: 46
Of these, on weekends: 0
Of these, on weekdays: 46

Breakdown by day of week:
day_of_week
Friday       10
Monday       28
Tuesday       6
Wednesday     2
Name: count, dtype: int64

Sample of non-holiday null dates:


,observation_date,day_of_week,is_weekend
58,2005-03-25,Friday,False
59,2005-03-28,Monday,False
84,2005-05-02,Monday,False
169,2005-08-29,Monday,False
255,2005-12-27,Tuesday,False
333,2006-04-14,Friday,False
334,2006-04-17,Monday,False
344,2006-05-01,Monday,False
429,2006-08-28,Monday,False
515,2006-12-26,Tuesday,False


#### Resolve empty values by forward fill

In [16]:
# Apply forward fill to handle missing values
tedrate_df_filled = tedrate_df.ffill()

# Verify no more null values
print(f"Null values before forward fill: {tedrate_df.isna().sum().sum()}")
print(f"Null values after forward fill: {tedrate_df_filled.isna().sum().sum()}")

# Save the updated CSV
tedrate_df_filled.to_csv("..\\datasets\\processed\\gfc_data\\processed_TEDRATE.csv", index=False)
print(f"\nUpdated CSV saved to: ..\\datasets\\processed\\gfc_data\\processed_TEDRATE.csv")

Null values before forward fill: 123
Null values after forward fill: 0

Updated CSV saved to: ..\datasets\processed\gfc_data\processed_TEDRATE.csv
